# Cross-validation, part 1: the tree models (forest and XGBoost), with and without the HMM

Stages 1 to 3 scored every model on **one** split: 121 training people, 30 test people. With only 30 people, a few unusual ones can move the score. Cross-validation uses **everyone**:

1. Deal all 151 people into 5 folds of about 30 people
2. For each fold: train on the other 4 folds (about 121 people), test on this fold
3. After 5 rounds, every person has been tested exactly once, by models that never saw them

So every comparison rests on 151 people instead of 30, and we also see how much the score moves from fold to fold.

**Same for every model:** the same folds, the same features, and an HMM learned again in each fold from that fold's *training* people only.

**GPU only.** scikit-learn's forest can only train on the CPU, so the forest here is XGBoost's random-forest mode on the RTX 3060. It is the same idea (many independent trees that vote) but a different implementation, so section 3 first checks it against the Stage 1 forest on the old split.

**Settings are not tuned again.** XGBoost uses the Stage 3 settings, the GPU forest gets a short check of tree depth on the Stage 1 training people (as the forest's leaf size was in Stage 1). Those settings were chosen on 121 of the 151 people, a small advantage the CNN shares (its design was also chosen on them).

**Kernel:** `.venv-tf` in this folder. The CNN is in `cv_cnn.ipynb`, and the comparison of all models is in `cv_compare.ipynb`.

## How this works: cross-validating the tree models, explained from scratch

The models themselves are explained in Stage 1 (random forest) and Stage 3 (XGBoost). This notebook re-tests them more thoroughly.

### Cross-validation, step by step

Stages 1 to 3 tested on **one** group of 30 people. With so few, a few unusual people can push a score up or down by luck. Cross-validation tests on **everyone**:

```
151 people, shuffled once (fixed seed), dealt into 5 folds of about 30

round 1:  [TEST] [train] [train] [train] [train]
round 2:  [train] [TEST] [train] [train] [train]
round 3:  [train] [train] [TEST] [train] [train]
round 4:  [train] [train] [train] [TEST] [train]
round 5:  [train] [train] [train] [train] [TEST]
```

In every round, a **brand-new model** is trained on 4 folds and tested on the 5th. After 5 rounds, every person has been tested exactly once, always by a model that never saw them.

What we get out:

| Result | What it tells us |
|---|---|
| **Pooled score** over all 151 people | A steadier estimate than the 30-person test |
| **Score in each fold** | How much the score moves just from *which* people are tested |
| **Score for each person** | Lets us compare two models person by person, on 151 people instead of 30 |

### Rules that keep it fair

| Rule | Why |
|---|---|
| Folds are made of **people**, not windows | Same reason as the test split: one person's windows look alike |
| All models use **the same folds** (`cv_folds()` in `common.py`) | Each model faces exactly the same test people |
| The HMM is learned again in each round, from that round's **training** people only | The HMM must not learn from the people it is tested on either |
| Settings are **not** tuned again | They came from earlier stages. Re-tuning inside every round would cost hours of GPU time. (Those settings were chosen on 121 of the 151 people, a small advantage every model shares) |
| Training on the **GPU only** | The no-CPU-training rule |

### Why the forest here is a "GPU forest"

scikit-learn's random forest (Stage 1) can only train on the CPU. XGBoost has a **random-forest mode** (`XGBRFClassifier`) that runs on the GPU. It works the same way: 200 trees, each trained on a random sample of windows (63%), each question choosing from a random 4 of the 16 features, then they vote.

The one real difference: XGBoost needs a **maximum depth** for its trees, while scikit-learn grows trees until the leaves are tiny. So:

1. Depths 8 and 12 are tried on the Stage 1 training people (depth 16 took about 9 minutes per forest, too much GPU heat for this check)
2. The GPU forest is trained on the old Stage 1 split and compared with the real Stage 1 forest, to see how good a stand-in it is


In [1]:
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_sample_weight

from common import (DATA_DIR, N_FOLDS, RESULTS_DIR, SEED, cv_folds, evaluate, learn_hmm, make_features, scores,
                    smooth, split_people)

# WHY: the rule is GPU only. Stop here instead of quietly training on the CPU.
assert xgb.build_info()['USE_CUDA'], 'this XGBoost cannot use the GPU'
print('XGBoost', xgb.__version__)

PROGRESS = DATA_DIR / 'cv_trees_progress.txt'
PROGRESS.write_text('')
def log(*parts):
    """Print, and also write to a file. WHY: a running notebook shows nothing until it ends, the file can be read any time."""
    line = ' '.join(str(p) for p in parts)
    print(line)
    with open(PROGRESS, 'a') as f:
        f.write(time.strftime('%H:%M:%S ') + line + '\n')

XGBoost 3.2.0


## 1. The folds

Shuffled with the fixed seed, so `cv_folds()` always gives the same folds. The CNN notebook uses the same function.

In [2]:
folds = cv_folds()
everyone = sorted(folds)
ages = pd.read_csv('../../data/open datasets/capture24/capture24/metadata.csv').set_index('pid')['age']
pd.DataFrame({'participant_id': everyone, 'fold': [folds[p] for p in everyone], 'age': ages[everyone].values}
             ).to_csv(RESULTS_DIR / 'cv_folds.csv', index=False)
print('people per fold and age group:')
print(pd.crosstab(ages[everyone].values, [folds[p] for p in everyone], margins=True).to_string())

people per fold and age group:
col_0   0   1   2   3   4  All
row_0                         
18-29   6  11  11   5  10   43
30-37   7   5  11   9   5   37
38-52  10   4   6  10   7   37
53+     8  10   2   6   8   34
All    31  30  30  30  30  151


## 2. Features for all 151 people

The same 16 features as Stages 1 and 3 (`make_features` in `common.py`, which Stage 3 checked against Stage 1). They are worked out once and kept in `processed/features_all.npz`, because every fold reuses them. This is not training, just arithmetic on each window.

In [3]:
FEATURE_FILE = DATA_DIR / 'features_all.npz'

if not FEATURE_FILE.exists():
    t0 = time.time()
    F, y, pid, start = [], [], [], []
    for p in everyone:                             # one person at a time, so the raw windows never all sit in RAM
        with np.load(DATA_DIR / f'{p}.npz') as f:
            F.append(make_features(f['X'])); y.append(f['y4'].astype(int)); start.append(f['start'])
        pid.append(np.full(len(y[-1]), p))
    np.savez(FEATURE_FILE, F=np.concatenate(F), y=np.concatenate(y), pid=np.concatenate(pid), start=np.concatenate(start))
    log(f'features worked out in {time.time() - t0:.0f} s')

with np.load(FEATURE_FILE) as f:
    F_all, y_all, pid_all, start_all = f['F'], f['y'], f['pid'], f['start']
assert list(np.unique(pid_all)) == everyone and F_all.shape == (len(y_all), 16)
with np.load(DATA_DIR / f'{everyone[77]}.npz') as f:    # the saved features match a fresh calculation
    assert np.allclose(F_all[pid_all == everyone[77]], make_features(f['X']))
fold_all = pd.Series(pid_all).map(folds).to_numpy()
print(f'{len(y_all):,} windows from {len(everyone)} people')
print('windows per fold:', np.bincount(fold_all).tolist())

301,164 windows from 151 people
windows per fold: [62343, 58225, 61030, 60154, 59412]


## 3. The models

**GPU forest** (`XGBRFClassifier`). Settings copied from a usual random forest:

- 200 trees, grown independently and averaged, nothing learned one after another
- each tree sees a random 63% of the windows (about what a forest's bootstrap sample holds)
- each split picks from a random 4 of the 16 features (the forest's usual "square root of the number of features")

scikit-learn's forest grows trees until the leaves are tiny; XGBoost needs a depth limit, so two depths are tried on the Stage 1 training people with person-grouped 5-fold validation, the same check as Stage 1. (Depth 16 was dropped: on this GPU one forest took about 9 minutes, so the check alone would have taken over an hour of GPU heat.)

**XGBoost:** exactly the Stage 3 model (depth 8, 200 rounds, learning rate 0.1).

Both get the same class weights (a rare moderate-vigorous window counts for more).

In [4]:
def make_rf(depth):
    return xgb.XGBRFClassifier(n_estimators=200, max_depth=depth, subsample=0.632, colsample_bynode=0.25,
                               objective='multi:softprob', tree_method='hist', device='cuda', n_jobs=4,
                               random_state=SEED)

def make_xgb():
    return xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, objective='multi:softprob',
                             tree_method='hist', device='cuda', n_jobs=4, random_state=SEED)

def fit(model, F, y):
    return model.fit(F, y, sample_weight=compute_sample_weight('balanced', y))

# check that both really train on the GPU
for model in [make_rf(4), make_xgb()]:
    model.set_params(n_estimators=5)
    config = fit(model, F_all[:5000], y_all[:5000]).get_booster().save_config().replace(' ', '')
    assert '"device":"cuda' in config, 'not training on the GPU'
log('training on the GPU: yes')

# choose the forest's depth on the Stage 1 training people
train_ids, test_ids = split_people()
in_train = np.isin(pid_all, train_ids)
F_tr, y_tr, pid_tr = F_all[in_train], y_all[in_train], pid_all[in_train]
rows = []
t0 = time.time()
for depth in [8, 12]:
    fold_f1 = []
    for fit_idx, val_idx in GroupKFold(5).split(F_tr, y_tr, pid_tr):
        model = fit(make_rf(depth), F_tr[fit_idx], y_tr[fit_idx])
        fold_f1.append(scores(y_tr[val_idx], model.predict(F_tr[val_idx]), 4)['macro-F1'])
    rows.append({'max_depth': depth, 'mean macro-F1': np.mean(fold_f1), 'std over folds': np.std(fold_f1)})
    log(rows[-1], f'{time.time() - t0:.0f} s')
rf_tuning = pd.DataFrame(rows)
rf_tuning.to_csv(RESULTS_DIR / 'cv_rf_gpu_tuning.csv', index=False)
rf_depth = int(rf_tuning.loc[rf_tuning['mean macro-F1'].idxmax(), 'max_depth'])
log(f'chosen depth {rf_depth}. (The Stage 1 forest got 0.748 in the same check.)')

training on the GPU: yes


C:\Users\syafi\SilverGuard Major Project\notebooks\DEEPDEEPDEEP_learning\.venv-tf\Lib\site-packages\xgboost\core.py:751: UserWarning: [16:59:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


{'max_depth': 8, 'mean macro-F1': np.float64(0.7278621347246148), 'std over folds': np.float64(0.02199788338023865)} 90 s


{'max_depth': 12, 'mean macro-F1': np.float64(0.7447526298266014), 'std over folds': np.float64(0.020456054722941756)} 878 s
chosen depth 12. (The Stage 1 forest got 0.748 in the same check.)


**How close is the GPU forest to the Stage 1 forest?** Train it on the Stage 1 training people and score it on the Stage 1 test people. If it is close, the forest results below stand in fairly for the Stage 1 forest.

In [5]:
in_test = np.isin(pid_all, test_ids)
stage1 = np.load(RESULTS_DIR / 'stage1_predictions.npz')
assert (y_all[in_test] == stage1['y']).all() and (pid_all[in_test] == stage1['pid']).all()   # same windows, same order

gpu_rf = fit(make_rf(rf_depth), F_tr, y_tr)
check = pd.DataFrame(evaluate('Stage 1 forest (CPU)', stage1['y'], stage1['pred_rf'])
                     + evaluate('GPU forest', stage1['y'], gpu_rf.predict(F_all[in_test])))
log(check.pivot(index='model', columns='task', values=['accuracy', 'macro-F1', 'kappa']).round(3).to_string())

                     accuracy        macro-F1          kappa       
task                  4-class binary  4-class binary 4-class binary
model                                                              
GPU forest              0.800  0.900    0.740  0.844   0.693  0.689
Stage 1 forest (CPU)    0.806  0.907    0.741  0.852   0.701  0.705


## 4. Cross-validation

For each of the 5 folds:

1. Train both models on the other 4 folds
2. Learn the HMM from the true labels of those same training people (so the HMM never sees the test fold either)
3. Predict the test fold, and smooth each test person's day with that HMM

Each round trains one forest and one XGBoost, a few minutes on the GPU.

In [6]:
probs = {'RF': np.zeros((len(y_all), 4)), 'XGB': np.zeros((len(y_all), 4))}
pred = {name: np.full(len(y_all), -1) for name in ['RF', 'RF+HMM', 'XGB', 'XGB+HMM']}
makers = {'RF': lambda: make_rf(rf_depth), 'XGB': make_xgb}
t0 = time.time()
for k in range(N_FOLDS):
    tr, te = fold_all != k, fold_all == k
    assert not set(pid_all[tr]) & set(pid_all[te])                  # nobody in both
    start_p, transition = learn_hmm(y_all[tr], pid_all[tr])         # this fold's HMM
    for name, make in makers.items():
        model = fit(make(), F_all[tr], y_all[tr])
        probs[name][te] = model.predict_proba(F_all[te])
        pred[name][te] = probs[name][te].argmax(axis=1)
        pred[name + '+HMM'][te] = smooth(probs[name][te], pid_all[te], start_p, transition)
    log(f'fold {k}: {te.sum():,} test windows from {len(np.unique(pid_all[te]))} people, {time.time() - t0:.0f} s')

assert all((p >= 0).all() for p in pred.values())              # every window predicted exactly once
np.savez(RESULTS_DIR / 'cv_trees_predictions.npz', pid=pid_all, start=start_all, y=y_all, fold=fold_all,
         probs_rf=probs['RF'], probs_xgb=probs['XGB'], **{f'pred_{n}': p for n, p in pred.items()})

fold 0: 62,343 test windows from 31 people, 256 s


fold 1: 58,225 test windows from 30 people, 471 s


fold 2: 61,030 test windows from 30 people, 629 s


fold 3: 60,154 test windows from 30 people, 788 s


fold 4: 59,412 test windows from 30 people, 938 s


## 5. A first look

All 151 people pooled, plus the spread over the 5 folds. The full comparison with the CNN, person by person, is in `cv_compare.ipynb`.

In [7]:
pooled = pd.DataFrame([r for n, p in pred.items() for r in evaluate(n, y_all, p)])
per_fold = pd.DataFrame([{'fold': k, **r} for k in range(N_FOLDS) for n, p in pred.items()
                         for r in evaluate(n, y_all[fold_all == k], p[fold_all == k])])
per_fold.to_csv(RESULTS_DIR / 'cv_trees_per_fold.csv', index=False)
order = list(pred)
print('all 151 people pooled:')
print(pooled.pivot(index='model', columns='task', values=['accuracy', 'macro-F1', 'kappa']).loc[order].round(3).to_string())
print('\nkappa in each fold:')
print(per_fold.pivot_table(index='model', columns=['task', 'fold'], values='kappa').loc[order].round(3).to_string())

all 151 people pooled:
        accuracy        macro-F1          kappa       
task     4-class binary  4-class binary 4-class binary
model                                                 
RF         0.803  0.900    0.749  0.862   0.707  0.723
RF+HMM     0.883  0.929    0.822  0.899   0.825  0.799
XGB        0.802  0.901    0.744  0.865   0.707  0.730
XGB+HMM    0.881  0.932    0.821  0.906   0.822  0.812

kappa in each fold:
task    4-class                             binary                            
fold          0      1      2      3      4      0      1      2      3      4
model                                                                         
RF        0.709  0.732  0.697  0.693  0.705  0.726  0.688  0.730  0.731  0.731
RF+HMM    0.821  0.848  0.820  0.815  0.819  0.801  0.778  0.810  0.805  0.793
XGB       0.706  0.732  0.695  0.692  0.708  0.737  0.693  0.739  0.738  0.731
XGB+HMM   0.816  0.854  0.817  0.810  0.813  0.814  0.795  0.823  0.824  0.798
